## Understanding lit function in Spark

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Understanding lit function in Spark").getOrCreate()

import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

24/02/25 16:53:09 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/02/25 16:53:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/02/25 16:53:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/02/25 16:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/02/25 16:53:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
from pyspark.sql.functions import lit

In [5]:
from pyspark.sql import Row

users_df = spark.createDataFrame(Row(**users) for users in users_list)
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [7]:
users_df.createOrReplaceTempView('vw_users')

In [8]:
spark.sql("""
    select id, amount_paid + 25.00 from vw_users
"""
).show()

+---+---------------------+
| id|(amount_paid + 25.00)|
+---+---------------------+
|  1|               148.45|
|  2|                 25.0|
|  3|                 25.0|
+---+---------------------+



In [9]:
users_df.select('id', 'amount_paid' + 25).show()

TypeError: can only concatenate str (not "int") to str

In [10]:
users_df.select('id', 'amount_paid' + lit(25)).show()

+---+------------------+
| id|(25 + amount_paid)|
+---+------------------+
|  1|              NULL|
|  2|              NULL|
|  3|              NULL|
+---+------------------+



In [11]:
users_df.select('id', 'amount_paid' + lit('25')).show()

+---+------------------+
| id|(25 + amount_paid)|
+---+------------------+
|  1|              NULL|
|  2|              NULL|
|  3|              NULL|
+---+------------------+



In [18]:
users_df.selectExpr('id', 'amount_paid' + lit('25')).show()

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.

In [13]:
lit('25')

Column<'25'>

In [14]:
lit(25)

Column<'25'>

In [22]:
from pyspark.sql.functions import col

users_df.select('id', col('amount_paid') + lit('25')).show()

+---+------------------+
| id|(amount_paid + 25)|
+---+------------------+
|  1|            148.45|
|  2|              25.0|
|  3|              25.0|
+---+------------------+

